# Chapter 3: Assigning Roles (Role Prompting)

**Technique:** Role prompting (persona prompting)

**Contents**
* [Lesson: Why Roles Matter](#lesson)
* [Exercises](#exercises)
* [Example Playground](#example-playground)

In [ ]:
# Make the project root importable from any working directory, then load the helpers.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lib" / "helpers.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib.helpers import get_completion, response_text, client, MODEL, MAX_TOKENS, SAMPLING_PARAMS

<a id="lesson"></a>
## Lesson: Why Roles Matter

When you add a role to the system prompt (for example, "You are a meticulous senior code reviewer"), Claude shifts its tone, focus, and depth of analysis to match that persona. This is not a trick; it mirrors how you'd brief a real colleague before asking for feedback.

**Why this matters in practice**

Without a role, Claude answers the question it's given but has no reason to push back, look for edge cases, or assume the worst. With a role, Claude adopts the priorities of that persona:

* A **security auditor** will flag injection vectors you didn't ask about.
* A **senior code reviewer** will notice boundary conditions even in a short snippet.
* An **SRE on call** will focus on operational impact over style.

**Where role prompting pays off most**

* **PR review**: "meticulous senior engineer who rejects off by one errors".
* **Security audit**: "application security engineer looking for injection and auth flaws".
* **Architecture critique**: "staff engineer who always asks about failure modes and scalability".
* **API design review**: "developer experience engineer who focuses on naming and consistency".

**How to structure it**

Put the role in the **system prompt**, not the user turn. System prompt roles persist across the whole conversation. User turn role instructions can be overridden or diluted as the conversation grows.

```python
SYSTEM_PROMPT = "You are a meticulous senior code reviewer who always checks boundary conditions."
response = get_completion("Is there a bug in this code?\n<code>...</code>", SYSTEM_PROMPT)
```

**The specificity rule**: "You are an expert" rarely helps, because Claude already knows about most topics. Specificity about *what the role cares about* is what changes behavior. "You are a senior reviewer who always checks boundary conditions and rejects functions that can raise an unhandled `IndexError`" is more useful than "You are a senior reviewer."

In [ ]:
# Same buggy snippet: see how the role changes the quality of feedback.
BUGGY = """def last_item(items):
    return items[len(items)]"""

print("WITHOUT ROLE:\n", get_completion(f"Is there a bug in this code?\n{BUGGY}"))
print("\nWITH ROLE:\n", get_completion(f"Is there a bug in this code?\n{BUGGY}", "You are a meticulous senior code reviewer who always checks boundary conditions."))

<a id="exercises"></a>
## Exercises

### Exercise 3.1: Catch the off by one with a reviewer role

**Scenario**: a teammate opens a PR claiming this utility function is correct:

```python
def last_item(items):
    return items[len(items)]
```

`items[len(items)]` always raises `IndexError`. Lists are indexed from 0, so the last element is at `items[len(items) - 1]` (or, idiomatically, `items[-1]`). Without a role, Claude may give a lukewarm response. Your job is to assign Claude a role (via `SYSTEM_PROMPT`) that makes it flag the code as incorrect.

**Task**: Replace `SYSTEM_PROMPT` with a role that causes Claude to scrutinize the code and catch the bug. Leave `PROMPT` unchanged.

**Success criteria**: the response contains at least one of: `incorrect`, `not correct`, `bug`, `off-by-one`, `off by one`, `IndexError`.

In [ ]:
from lib.grading import includes_any, grade

SYSTEM_PROMPT = "[Replace this text]"
PROMPT = "A teammate says this returns the last element. Is it correct?\ndef last_item(items):\n    return items[len(items)]"

response = get_completion(PROMPT, SYSTEM_PROMPT)

def grade_exercise(text):
    return includes_any(text, ["incorrect", "not correct", "bug", "off-by-one", "off by one", "IndexError"])

grade(response, grade_exercise(response))

In [ ]:
from hints import exercise_3_1_hint
print(exercise_3_1_hint)

## Common mistakes, stretch challenge, and reflection

**Common mistakes**

* **Role too generic**: "You are an expert developer" gives Claude a persona but no explicit priorities. A more effective role specifies *what the reviewer cares about*: "You are a senior engineer who rejects any function that can raise an unhandled `IndexError` or silently return `None`." The more targeted the role, the more consistent the output.
* **Role in the user turn instead of system**: placing "Act as a code reviewer" inside the user message works for a single call, but in a conversation with many turns it can drift. System prompt roles are stable across all turns.
* **No domain context**: "senior engineer" is generic. "senior engineer reviewing Python services where unhandled exceptions page the on-call" gives Claude operating constraints, not just a title.

**Stretch challenge**

Extend `SYSTEM_PROMPT` so Claude not only flags the bug but also returns the corrected function. Confirm the response includes `items[-1]`, `len(items) - 1`, or an equivalent fix (and ideally handles the empty list case).

**Reflect**: Which types of code review tasks benefit most from role prompting, and which don't? Consider: stylistic opinions (formatting, naming) vs correctness errors (`IndexError`, race conditions) vs security flaws (injection, unvalidated input). Where would you inject a role prompted Claude call into your team's existing CI/PR pipeline?

<a id="example-playground"></a>
## Example Playground

Use the cell below to experiment with different roles on the same prompt. Try assigning an SRE, an application security engineer, or an API designer, then observe how the feedback changes.

In [ ]:
# Swap the role and see how the analysis shifts
SYSTEM_PROMPT = "You are an application security engineer doing a defensive code review before release. Treat every input as untrusted, and for each issue explain the risk and the fix."
PROMPT = """Review this Flask route handler:
@app.get("/user")
def get_user():
    user_id = request.args["id"]
    rows = db.execute("SELECT * FROM users WHERE id=" + user_id).fetchall()
    return jsonify(rows)"""
print(get_completion(PROMPT, SYSTEM_PROMPT))